# Lightweight Interpretable Nutrient Analysis Model - Demo

This notebook demonstrates the complete pipeline:
1. Model architecture
2. Training
3. Interpretability features
4. Mobile optimization
5. Evaluation

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from src.models.nutrient_model import NutrientAnalysisModel
from src.data.preprocessing import preprocess_image
from src.interpretability.gradcam import GradCAM
from src.interpretability.lime_explainer import LIMEExplainer
from src.optimization.quantization import quantize_model

## 1. Model Creation

In [ ]:
# Create model
model = NutrientAnalysisModel(
    num_classes=500,
    width_multiplier=1.0,
    use_se=True,
    use_sa=True,
    dropout=0.2
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"Estimated size: {total_params * 4 / (1024**2):.2f} MB")

## 2. Inference Demo

In [ ]:
# Create dummy image (replace with actual food image)
dummy_image = Image.new('RGB', (224, 224), color=(128, 128, 128))

# Preprocess
input_tensor = preprocess_image(dummy_image)

# Inference
model.eval()
with torch.no_grad():
    predictions = model(input_tensor)

print("Predictions:")
for key, value in predictions.items():
    print(f"  {key}: {value.shape}")

## 3. Interpretability - Grad-CAM

In [ ]:
# Create Grad-CAM explainer
gradcam = GradCAM(model, target_layer='features.16')

# Generate heatmap
heatmap = gradcam.generate_heatmap(input_tensor)

# Visualize
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(dummy_image)
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(heatmap, cmap='jet')
plt.title('Grad-CAM Heatmap')
plt.axis('off')

plt.subplot(1, 3, 3)
overlay, _ = gradcam.visualize(input_tensor, dummy_image)
plt.imshow(overlay)
plt.title('Overlay')
plt.axis('off')

plt.tight_layout()
plt.show()

## 4. Model Quantization

In [ ]:
# Apply dynamic quantization
quantized_model = quantize_model(model, quantization_type='dynamic')

# Compare sizes
original_size = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024**2)
quantized_size = sum(p.numel() * p.element_size() for p in quantized_model.parameters()) / (1024**2)

print(f"Original model size: {original_size:.2f} MB")
print(f"Quantized model size: {quantized_size:.2f} MB")
print(f"Compression ratio: {original_size/quantized_size:.2f}x")

## 5. Performance Summary

In [ ]:
print("Model Performance Summary (from paper):")
print("="*50)
print(f"Top-1 Accuracy: 97.1%")
print(f"Top-5 Accuracy: 98.0% (with cross-validation)")
print(f"MAE (Nutrients): 7.2%")
print(f"Model Size: 11.0 MB")
print(f"Inference Time (Lab): 150 ms")
print(f"Inference Time (Real): 240-310 ms")
print(f"Energy Consumption: 180 mJ")
print("\nFood Security Categories:")
print(f"  Staple Foods: 94.1% accuracy")
print(f"  Affordable Proteins: 93.2% accuracy")
print(f"  Accessible Produce: 92.8% accuracy")